In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import gzip
import warnings
warnings.filterwarnings('ignore')
from sklearn import preprocessing
from sklearn import metrics
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence
from sklearn.model_selection import train_test_split
import seaborn as sns

%matplotlib inline

print("Loading data...")

columns = [
    'karl_id', 'host_name', 'model_name', 'hardware_make', 'karl_last_seen',
    'auth_username', 'serial_number', 'group_id', 'tenant_id', 'platform',
    'metric_category', 'measure_name', 'time', 'p90_processor_time',
    'avg_processor_time', 'max_cpu_usage', 'p90_memory_utilization',
    'avg_memory_utilization', 'max_memory_usage', 'p10_battery_health',
    'avg_battery_health', 'cpu_count', 'memory_count', 'memory_size_gb',
    'driver_vendor', 'os', 'wifi_mac_add', 'driver_version', 'driver_date',
    'os_version', 'driver', 'agent_id', 'performance_status', 'device_status',
    'max_battery_temperature', 'avg_battery_temperature', 'p90_battery_temperature',
    'avg_cpu_temp', 'p90_cpu_temp', 'avg_battery_discharge', 'p90_battery_discharge',
    'avg_boot_time', 'p90_boot_time', 'uptime_days', 'total_app_crash'
]

# Load sample
chunk_size = 1000000
sample_data = []
with gzip.open('000.gz', 'rt') as f:
    for i, chunk in enumerate(pd.read_csv(f, sep='|', names=columns, chunksize=chunk_size)):
        sample_data.append(chunk)
        if i >= 4:
            break

df = pd.concat(sample_data, ignore_index=True)

numeric_cols = [
    'avg_processor_time', 'max_cpu_usage', 'avg_memory_utilization',
    'max_memory_usage', 'avg_battery_health', 'cpu_count', 'memory_size_gb',
    'avg_cpu_temp', 'avg_boot_time', 'p90_boot_time',
    'uptime_days', 'total_app_crash'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Loaded {len(df)} rows")

    

Loading data...
Loaded 5000000 rows


In [ ]:
# making a correlation matrix -> I think this is done before the loop?
# maybe should be done w/ the 90th percentile?

print(df.columns)

feature_cols = ['avg_processor_time','max_cpu_usage','avg_memory_utilization', 'max_memory_usage',
'avg_battery_health','cpu_count','memory_size_gb','avg_cpu_temp','avg_boot_time','p90_boot_time',
'uptime_days', 'total_app_crash']

dfcol = df[feature_cols]

print(dfcol.columns)

corr = dfcol.corr()
sm.graphics.plot_corr(corr, xnames=list(corr.columns))
plt.show()

print (corr)

sns.heatmap(dfcol.corr(), vmax=.8, square=True,annot=True,fmt='.1f')

Index(['karl_id', 'host_name', 'model_name', 'hardware_make', 'karl_last_seen',
       'auth_username', 'serial_number', 'group_id', 'tenant_id', 'platform',
       'metric_category', 'measure_name', 'time', 'p90_processor_time',
       'avg_processor_time', 'max_cpu_usage', 'p90_memory_utilization',
       'avg_memory_utilization', 'max_memory_usage', 'p10_battery_health',
       'avg_battery_health', 'cpu_count', 'memory_count', 'memory_size_gb',
       'driver_vendor', 'os', 'wifi_mac_add', 'driver_version', 'driver_date',
       'os_version', 'driver', 'agent_id', 'performance_status',
       'device_status', 'max_battery_temperature', 'avg_battery_temperature',
       'p90_battery_temperature', 'avg_cpu_temp', 'p90_cpu_temp',
       'avg_battery_discharge', 'p90_battery_discharge', 'avg_boot_time',
       'p90_boot_time', 'uptime_days', 'total_app_crash'],
      dtype='object')
Index(['avg_processor_time', 'max_cpu_usage', 'avg_memory_utilization',
       'max_memory_usage', 'avg_

In [ ]:
# use the list to select a subset from original DataFrame
independent_variables = ['avg_processor_time','max_cpu_usage','avg_memory_utilization', 'max_memory_usage',
'avg_battery_health','cpu_count','memory_size_gb','avg_cpu_temp','avg_boot_time','p90_boot_time',
'uptime_days']

X = dfcol[independent_variables]
y = dfcol['total_app_crash']

X = X.replace([np.inf, -np.inf], np.nan).dropna() # from chat check

thresh = 10

for i in np.arange(0,len(independent_variables)):
    vif = [variance_inflation_factor(X[independent_variables].values, ix) for ix in range(X[independent_variables].shape[1])]
    maxloc = vif.index(max(vif))
    if max(vif) > thresh:
        print("vif :", vif)
        print('dropping \'' + X[independent_variables].columns[maxloc] + '\' at index: ' + str(maxloc))
        del independent_variables[maxloc]
    else:
        break

print('Final variables:', independent_variables)

In [ ]:
# final variables from VIF are ['max_cpu_usage', 'avg_memory_utilization', 'cpu_count', 'memory_size_gb', 'avg_boot_time', 'p90_boot_time', 'uptime_days']

# graphing variables